In [ ]:
!pip install unsloth trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 M

In [ ]:
# All Imports
import os

os.environ.setdefault("UNSLOTH_RETURN_LOGITS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("unsloth").setLevel(logging.ERROR)
logging.getLogger("unsloth_zoo").setLevel(logging.ERROR)

import re
import json
import time
import math
import random
import zipfile
from typing import Any, Dict, List, Tuple, Optional
from collections import defaultdict

import torch
import torch.nn.functional as F
from PIL import Image
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from tqdm import tqdm

try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as AutoModelForVision2SeqCompat
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoModelForVision2SeqCompat

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

In [ ]:
# Config

# --- SLAKE dataset location -------------------------------------------------
SLAKE_REPO_ID = "BoKelvin/SLAKE"
SLAKE_TRAIN_JSON = "train.json"
SLAKE_TEST_JSON = "test.json"
SLAKE_IMGS_ZIP = "imgs.zip"
SLAKE_LOCAL_DIR = "/content/slake_data"

TEACHER_MODEL_ID = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"
STUDENT_MODEL_ID = "unsloth/Qwen2.5-VL-3B-Instruct"

OUTPUT_DIR = "/content/causal_teacher_student_slake"
ROUND_CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "round_checkpoints")
LOG_PATH = os.path.join(OUTPUT_DIR, "round_log.jsonl")
EVAL_PATH = os.path.join(OUTPUT_DIR, "eval_log.jsonl")
STATE_PATH = os.path.join(OUTPUT_DIR, "state.json")
TRAIN_RECORDS_STATE_PATH = os.path.join(OUTPUT_DIR, "train_records_state.json")

MAX_TRAIN_SAMPLES: Optional[int] = None
MAX_TEST_SAMPLES: Optional[int] = None

WARM_START_FRACTION = 0.10
WARM_START_MIN_EXAMPLES = 100

SELFGEN_SUBSAMPLE_FRACTION = 1.0

NUM_SELF_TRAIN_ROUNDS = 3
KEEP_THRESHOLD = 0.55

MAX_SEQ_LENGTH = 2048
MAX_STEPS_PER_ROUND = 60
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-4
LORA_R = 32
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 5

SAVE_STEPS_PER_ROUND = 10
SAVE_TOTAL_LIMIT_PER_ROUND = 2

GENERATION_BATCH_SIZE = 16
GENERATION_MAX_NEW_TOKENS = 256
GENERATION_TEMPERATURE = 0.4
GENERATION_DO_SAMPLE = True

TEACHER_MAX_NEW_TOKENS = 192
TEACHER_TEMPERATURE = 0.2
TEACHER_DO_SAMPLE = False
TEACHER_GENERATION_BATCH_SIZE = 8

USE_KL_DISTILL_LOSS = True
KL_LOSS_WEIGHT = 0.5        
KL_MAX_ALIGN_TOKENS = 64 

RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# UTILS

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def unwrap_text(x: Any) -> str:
    if x is None:
        return ""
    try:
        if isinstance(x, float) and math.isnan(x):
            return ""
    except Exception:
        pass
    if isinstance(x, list):
        if len(x) == 0:
            return ""
        return unwrap_text(x[0])
    if isinstance(x, dict):
        for k in ["answer", "text", "label", "value"]:
            if k in x:
                return unwrap_text(x[k])
    return str(x)


def normalize_text(s: Any) -> str:
    s = unwrap_text(s).lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def normalize_question(s: Any) -> str:
    s = unwrap_text(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def normalize_answer(s: Any) -> str:
    s = unwrap_text(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def answer_to_text(ans: Any) -> str:
    return unwrap_text(ans).strip()


def safe_json_loads(text: str) -> Optional[Any]:
    try:
        return json.loads(text)
    except Exception:
        return None


def extract_json_object(text: str) -> Optional[dict]:
    if not text:
        return None
    obj = safe_json_loads(text.strip())
    if isinstance(obj, dict):
        return obj

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    obj = safe_json_loads(m.group(0))
    if isinstance(obj, dict):
        return obj
    return None


def strip_step_prefix(step: str) -> str:
    step = step.strip()
    step = re.sub(r"^\s*STEP\s*\d+\s*:\s*", "", step, flags=re.IGNORECASE)
    step = re.sub(r"^\s*\d+\s*[\).\-\:]\s*", "", step)
    return step.strip()


def extract_steps(text: str, stop_keyword: str = "ANSWER") -> List[str]:
    if not text:
        return []

    pattern = r"\b" + re.escape(stop_keyword) + r"\s*:"
    reasoning_part = re.split(pattern, text, maxsplit=1, flags=re.IGNORECASE)[0]

    lines = [ln.rstrip() for ln in reasoning_part.splitlines()]
    steps: List[str] = []
    current: List[str] = []
    saw_step_header = False

    for raw in lines:
        line = raw.strip()
        if not line:
            continue
        if re.match(r"^\s*STEP\s*\d+\s*:", line, flags=re.IGNORECASE):
            saw_step_header = True
            if current:
                steps.append(" ".join(current).strip())
                current = []
            current.append(strip_step_prefix(line))
        else:
            if saw_step_header and current:
                current.append(line)

    if current:
        steps.append(" ".join(current).strip())

    steps = [s for s in (strip_step_prefix(s) for s in steps) if s]
    return steps


def extract_answer(text: str) -> str:
    if not text:
        return ""

    m = re.search(r"\bANSWER\s*:\s*(.*)$", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        answer = m.group(1).strip()
        answer = answer.splitlines()[0].strip()
        return answer

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if lines:
        return lines[-1]
    return text.strip()


def extract_quality(text: str, default: float = 0.0) -> float:
    if not text:
        return default

    m = re.search(r"\bQUALITY\s*:\s*([0-9]*\.?[0-9]+)", text, flags=re.IGNORECASE)
    if not m:
        return default

    try:
        val = float(m.group(1))
    except ValueError:
        return default

    return max(0.0, min(1.0, val))


def format_steps(steps: List[str], answer: str) -> str:
    if not steps:
        steps = ["Inspect the image and answer the question from visible evidence."]
    out = []
    for i, step in enumerate(steps, start=1):
        step = strip_step_prefix(step)
        if not step:
            continue
        out.append(f"STEP {i}: {step}")
    out.append(f"ANSWER: {answer.strip()}")
    return "\n".join(out)


def minimal_template(answer: str) -> str:
    return format_steps(
        ["Inspect the image and use only visible evidence to answer the question."],
        answer,
    )


YES_NO_ANSWERS = {"yes", "no"}


def is_closed_answer(answer: str) -> bool:
    return normalize_answer(answer) in YES_NO_ANSWERS


def make_output_dirs() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(ROUND_CHECKPOINT_DIR, exist_ok=True)


def log_jsonl(path: str, obj: dict) -> None:
    make_output_dirs()
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def set_generation_padding(processor, side: str = "left") -> None:
    tok = getattr(processor, "tokenizer", None)
    if tok is None:
        return
    tok.padding_side = side
    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token


def disable_max_length_conflict(model) -> None:
    for obj in (
        model,
        getattr(model, "base_model", None),
        getattr(model, "base_model", None) and getattr(model.base_model, "model", None),
    ):
        if obj is None:
            continue
        gen_cfg = getattr(obj, "generation_config", None)
        if gen_cfg is not None and hasattr(gen_cfg, "max_length"):
            gen_cfg.max_length = None


def default_state() -> Dict[str, Any]:
    return {
        "warm_start_done": False,
        "refine_done_round": -1,  
        "completed_rounds": [],     
        "final_eval_done": False,
    }


def load_state() -> Dict[str, Any]:
    if not os.path.isfile(STATE_PATH):
        return default_state()
    try:
        with open(STATE_PATH, "r", encoding="utf-8") as f:
            state = json.load(f)
        merged = default_state()
        merged.update(state)
        return merged
    except Exception:
        print(f"WARNING: could not parse {STATE_PATH}; starting fresh state.")
        return default_state()


def save_state(state: Dict[str, Any]) -> None:
    make_output_dirs()
    tmp_path = STATE_PATH + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)
    os.replace(tmp_path, STATE_PATH)


def save_train_records(records: List[Dict[str, Any]]) -> None:
    make_output_dirs()
    tmp_path = TRAIN_RECORDS_STATE_PATH + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(records, f)
    os.replace(tmp_path, TRAIN_RECORDS_STATE_PATH)


def load_train_records_state() -> Optional[List[Dict[str, Any]]]:
    if not os.path.isfile(TRAIN_RECORDS_STATE_PATH):
        return None
    with open(TRAIN_RECORDS_STATE_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
# Data Loading

def download_slake_files(local_dir: str = SLAKE_LOCAL_DIR) -> Dict[str, str]:
    os.makedirs(local_dir, exist_ok=True)

    paths: Dict[str, str] = {}
    for fname in (SLAKE_TRAIN_JSON, SLAKE_TEST_JSON, SLAKE_IMGS_ZIP):
        paths[fname] = hf_hub_download(
            repo_id=SLAKE_REPO_ID,
            repo_type="dataset",
            filename=fname,
            local_dir=local_dir,
        )

    imgs_dir = os.path.join(local_dir, "imgs")
    if not os.path.isdir(imgs_dir) or not os.listdir(imgs_dir):
        print(f"Extracting {paths[SLAKE_IMGS_ZIP]} -> {local_dir} ...")
        with zipfile.ZipFile(paths[SLAKE_IMGS_ZIP], "r") as zf:
            zf.extractall(local_dir)

    paths["imgs_dir"] = imgs_dir
    return paths


def load_image(path: str):
    img = Image.open(path)
    return img.convert("RGB")


def get_image(rec: Dict[str, Any]):
    """Lazily opens the PIL image for a record from disk. Records only
    store `image_path` (a plain string) rather than a decoded PIL image,
    which keeps them JSON-serializable for the checkpoint/resume state
    files -- see section 2B."""
    return load_image(rec["image_path"])


def build_records(json_path: str, imgs_dir: str, max_rows: Optional[int] = None) -> List[Dict[str, Any]]:
    with open(json_path, "r", encoding="utf-8") as f:
        raw_items = json.load(f)

    records: List[Dict[str, Any]] = []

    for item in raw_items:
        if str(item.get("q_lang", "en")).strip().lower() != "en":
            continue

        question = unwrap_text(item.get("question", ""))
        answer = answer_to_text(item.get("answer", ""))
        img_name = item.get("img_name", "")

        if not question or not answer or not img_name:
            continue

        img_path = os.path.join(imgs_dir, img_name)
        if not os.path.isfile(img_path):
            continue

        answer_type = str(item.get("answer_type", "")).strip().upper()
        if answer_type not in {"OPEN", "CLOSED"}:
            answer_type = "CLOSED" if is_closed_answer(answer) else "OPEN"

        records.append(
            {
                "record_id": len(records),
                "image_path": img_path,
                "question": question,
                "answer": answer,
                "answer_norm": normalize_answer(answer),
                "question_norm": normalize_question(question),
                "answer_type": answer_type,

                "target_reasoning": "",
                "target_answer": answer,
                "target_source": "unset",

                "teacher_seed_reasoning": "",
                "teacher_refined_reasoning": "",

                "student_reasoning": "",
                "student_answer": "",

                "teacher_quality": None,
            }
        )

        if max_rows is not None and len(records) >= max_rows:
            break

    if not records:
        raise RuntimeError(f"No usable English-language records found in {json_path}.")

    return records


# Evaluation Metrics

def exact_match(pred: str, gt: str) -> int:
    return int(normalize_answer(pred) == normalize_answer(gt))


def _tokenize(s: str) -> List[str]:
    norm = normalize_answer(s)
    return norm.split() if norm else []


def token_overlap_recall(pred: str, gt: str) -> float:
    gt_tokens = _tokenize(gt)
    if not gt_tokens:
        return 0.0
    pred_tokens = set(_tokenize(pred))
    hits = sum(1 for t in gt_tokens if t in pred_tokens)
    return hits / len(gt_tokens)


def compute_bleu1(pred: str, gt: str) -> float:
    pred_tokens = _tokenize(pred)
    gt_tokens = _tokenize(gt)

    if not pred_tokens or not gt_tokens:
        return 0.0

    from collections import Counter
    pred_counts = Counter(pred_tokens)
    gt_counts = Counter(gt_tokens)

    overlap = sum(min(cnt, gt_counts.get(tok, 0)) for tok, cnt in pred_counts.items())
    precision = overlap / len(pred_tokens)

    c = len(pred_tokens)
    r = len(gt_tokens)
    bp = 1.0 if c > r else (float(math.exp(1 - r / c)) if c > 0 else 0.0)

    return bp * precision


def _lcs_len(a: List[str], b: List[str]) -> int:
    n, m = len(a), len(b)
    if n == 0 or m == 0:
        return 0
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]


def _rouge_n(pred_tokens: List[str], gt_tokens: List[str], n: int) -> float:
    def ngrams(tokens, n):
        return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

    pred_ngrams = ngrams(pred_tokens, n)
    gt_ngrams = ngrams(gt_tokens, n)
    if not pred_ngrams or not gt_ngrams:
        return 0.0

    from collections import Counter
    pred_counts = Counter(pred_ngrams)
    gt_counts = Counter(gt_ngrams)
    overlap = sum(min(cnt, gt_counts.get(ng, 0)) for ng, cnt in pred_counts.items())

    precision = overlap / len(pred_ngrams)
    recall = overlap / len(gt_ngrams)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def compute_rouge(pred: str, gt: str) -> Tuple[float, float, float]:
    pred_tokens = _tokenize(pred)
    gt_tokens = _tokenize(gt)

    r1 = _rouge_n(pred_tokens, gt_tokens, 1)
    r2 = _rouge_n(pred_tokens, gt_tokens, 2)

    if not pred_tokens or not gt_tokens:
        rL = 0.0
    else:
        lcs = _lcs_len(pred_tokens, gt_tokens)
        precision = lcs / len(pred_tokens)
        recall = lcs / len(gt_tokens)
        rL = 0.0 if (precision + recall == 0) else 2 * precision * recall / (precision + recall)

    return r1, r2, rL

In [ ]:
# Prompts

SYSTEM_REASONING = (
    "You are a medical imaging assistant. "
    "Generate concise stepwise reasoning grounded only in the image. "
    "Use the exact format requested."
)

SYSTEM_TEACHER_SEED = (
    "You are a senior medical imaging teacher. "
    "Given the image, question, and known correct answer, write a short causal "
    "stepwise explanation that supports the answer. "
    "Keep it concise and grounded only in visible evidence."
)

SYSTEM_TEACHER_CRITIC = (
    "You are a causal critic for medical image reasoning. "
    "Given the image, question, known correct answer, and a student's numbered reasoning, "
    "prune redundant or unsupported steps and write a corrected reasoning chain using "
    "only evidence visible in the image. "
    "If a step cannot be confidently corrected, replace it with UNVERIFIED. "
    "Do not use JSON. Reply only in the plain text format requested."
)


def build_student_messages(image, question: str, target_reasoning: Optional[str] = None,
                           target_answer: Optional[str] = None) -> List[Dict[str, Any]]:
    if target_reasoning is None:
        target_reasoning = ""
    if target_answer is None:
        target_answer = ""

    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM_REASONING},
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": (
                        "Question: {q}\n\n"
                        "Write 2 to 4 short numbered reasoning steps, then the final answer.\n"
                        "Use exactly this format:\n"
                        "STEP 1: ...\n"
                        "STEP 2: ...\n"
                        "ANSWER: ...\n"
                    ).format(q=question),
                },
            ],
        },
    ]

    if target_reasoning.strip() or target_answer.strip():
        assistant_text = format_steps(
            extract_steps(target_reasoning) if target_reasoning.strip() else [],
            target_answer.strip(),
        )
        messages.append(
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": assistant_text},
                ],
            }
        )

    return messages


def build_teacher_seed_messages(image, question: str, answer: str) -> List[Dict[str, Any]]:
    return [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM_TEACHER_SEED},
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": (
                        "Question: {q}\n"
                        "Known correct answer: {a}\n\n"
                        "Write a short numbered reasoning chain that supports the known correct answer. "
                        "Use 2 to 4 steps. End with:\n"
                        "ANSWER: {a}\n"
                    ).format(q=question, a=answer),
                },
            ],
        },
    ]


def build_teacher_critic_messages(image, question: str, answer: str, student_reasoning: str) -> List[Dict[str, Any]]:
    return [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM_TEACHER_CRITIC},
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": (
                        "Question: {q}\n"
                        "Known correct answer: {a}\n\n"
                        "Student reasoning:\n"
                        "{r}\n\n"
                        "Task:\n"
                        "1) Keep only steps that are causally necessary and supported by the image.\n"
                        "2) Remove redundant steps.\n"
                        "3) Correct unsupported steps using only the image; use UNVERIFIED if unsure.\n"
                        "4) Reply using EXACTLY this format, nothing else:\n"
                        "STEP 1: ...\n"
                        "STEP 2: ...\n"
                        "QUALITY: <a number from 0 to 1 rating how good the STUDENT's original "
                        "reasoning was, where 1 means fully correct and well-supported>\n"
                    ).format(q=question, a=answer, r=student_reasoning.strip() if student_reasoning.strip() else "(no reasoning)")
                },
            ],
        },
    ]

In [ ]:
# Teacher/Critique Model

class TeacherVLM:
    def __init__(self, model_id: str, device: str = DEVICE):
        self.device = device
        self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

        self.model = AutoModelForVision2SeqCompat.from_pretrained(
            model_id,
            torch_dtype=dtype,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True,
        )
        self.model.eval()

        set_generation_padding(self.processor, "left")
        disable_max_length_conflict(self.model)

    @torch.no_grad()
    def generate_text(
        self,
        messages: List[Dict[str, Any]],
        image,
        max_new_tokens: int = TEACHER_MAX_NEW_TOKENS,
        temperature: float = TEACHER_TEMPERATURE,
        do_sample: bool = TEACHER_DO_SAMPLE,
    ) -> str:
        texts = self.generate_text_batch(
            [messages], [image], max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=do_sample,
        )
        return texts[0]

    @torch.no_grad()
    def generate_text_batch(
        self,
        messages_list: List[List[Dict[str, Any]]],
        images: List[Any],
        max_new_tokens: int = TEACHER_MAX_NEW_TOKENS,
        temperature: float = TEACHER_TEMPERATURE,
        do_sample: bool = TEACHER_DO_SAMPLE,
    ) -> List[str]:
        prompts = [
            self.processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in messages_list
        ]

        inputs = self.processor(
            text=prompts,
            images=images,
            return_tensors="pt",
            padding=True,
        )

        for k, v in list(inputs.items()):
            if torch.is_tensor(v):
                inputs[k] = v.to(self.model.device)

        gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample)
        if do_sample:
            gen_kwargs["temperature"] = temperature

        output_ids = self.model.generate(**inputs, **gen_kwargs)
        input_len = inputs["input_ids"].shape[1]

        results = []
        for ids in output_ids:
            text = self.processor.decode(ids[input_len:], skip_special_tokens=True)
            results.append(text.strip())
        return results

    def generate_seed_reasoning(self, image, question: str, answer: str) -> str:
        messages = build_teacher_seed_messages(image, question, answer)
        raw = self.generate_text(messages, image=image)
        steps = extract_steps(raw)
        if not steps:
            steps = [
                "Observe the visible findings in the image that are relevant to the question.",
                "Use those findings to support the known correct answer.",
            ]
        return format_steps(steps[:4], answer)

    def generate_seed_reasoning_batch(
        self, images: List[Any], questions: List[str], answers: List[str]
    ) -> List[str]:
        messages_list = [
            build_teacher_seed_messages(img, q, a)
            for img, q, a in zip(images, questions, answers)
        ]
        raws = self.generate_text_batch(messages_list, images)

        results = []
        for raw, answer in zip(raws, answers):
            steps = extract_steps(raw)
            if not steps:
                steps = [
                    "Observe the visible findings in the image that are relevant to the question.",
                    "Use those findings to support the known correct answer.",
                ]
            results.append(format_steps(steps[:4], answer))
        return results

    def _parse_critique(self, raw: str, student_reasoning: str) -> Dict[str, Any]:
        quality = extract_quality(raw, default=0.0)
        refined_steps = extract_steps(raw, stop_keyword="QUALITY")

        if not refined_steps:
            refined_steps = extract_steps(student_reasoning)
            if not refined_steps:
                refined_steps = [
                    "Inspect the image carefully for evidence relevant to the question."
                ]

        refined_steps = [strip_step_prefix(s) for s in refined_steps if strip_step_prefix(s)]
        refined_steps = refined_steps[:4]

        return {
            "quality_score": quality,
            "refined_steps": refined_steps,
            "removed_step_ids": [],
            "notes": raw[:500],
            "raw": raw,
        }

    def refine_reasoning(self, image, question: str, answer: str, student_reasoning: str) -> Dict[str, Any]:
        messages = build_teacher_critic_messages(image, question, answer, student_reasoning)
        raw = self.generate_text(
            messages,
            image=image,
            max_new_tokens=TEACHER_MAX_NEW_TOKENS,
            temperature=TEACHER_TEMPERATURE,
            do_sample=TEACHER_DO_SAMPLE,
        )
        return self._parse_critique(raw, student_reasoning)

    def refine_reasoning_batch(
        self,
        images: List[Any],
        questions: List[str],
        answers: List[str],
        student_reasonings: List[str],
    ) -> List[Dict[str, Any]]:
        messages_list = [
            build_teacher_critic_messages(img, q, a, sr)
            for img, q, a, sr in zip(images, questions, answers, student_reasonings)
        ]
        raws = self.generate_text_batch(
            messages_list,
            images,
            max_new_tokens=TEACHER_MAX_NEW_TOKENS,
            temperature=TEACHER_TEMPERATURE,
            do_sample=TEACHER_DO_SAMPLE,
        )
        return [
            self._parse_critique(raw, sr)
            for raw, sr in zip(raws, student_reasonings)
        ]

In [ ]:
# Student Generation and Refinement

@torch.no_grad()
def student_generate_batch(model, processor, records: List[Dict[str, Any]], indices: List[int]) -> Dict[int, str]:
    if hasattr(FastVisionModel, "for_inference"):
        FastVisionModel.for_inference(model)
    else:
        model.eval()

    set_generation_padding(processor, "left")

    outputs: Dict[int, str] = {}

    for start in tqdm(range(0, len(indices), GENERATION_BATCH_SIZE), desc="Student generating"):
        batch_ids = indices[start:start + GENERATION_BATCH_SIZE]
        batch = [records[i] for i in batch_ids]

        texts = []
        images = []
        for rec in batch:
            img = get_image(rec)
            msgs = build_student_messages(img, rec["question"])
            prompt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            texts.append(prompt)
            images.append(img)

        inputs = processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
        )

        for k, v in list(inputs.items()):
            if torch.is_tensor(v):
                inputs[k] = v.to(model.device)

        gen_kwargs = dict(max_new_tokens=GENERATION_MAX_NEW_TOKENS, do_sample=GENERATION_DO_SAMPLE)
        if GENERATION_DO_SAMPLE:
            gen_kwargs["temperature"] = GENERATION_TEMPERATURE

        output_ids = model.generate(**inputs, **gen_kwargs)
        input_len = inputs["input_ids"].shape[1]

        for rec, ids in zip(batch, output_ids):
            decoded = processor.decode(ids[input_len:], skip_special_tokens=True).strip()
            outputs[rec["record_id"]] = decoded

    set_generation_padding(processor, "right")

    if hasattr(FastVisionModel, "for_training"):
        FastVisionModel.for_training(model)
    else:
        model.train()

    return outputs


def refine_training_targets(
    records: List[Dict[str, Any]],
    model,
    processor,
    teacher: TeacherVLM,
    round_idx: int,
) -> Dict[str, Any]:
    n = len(records)
    k = max(1, int(round(n * SELFGEN_SUBSAMPLE_FRACTION)))

    rng = random.Random(RANDOM_SEED + round_idx)
    selected = sorted(rng.sample(range(n), k=k))

    generated = student_generate_batch(model, processor, records, selected)

    keep_count = 0
    fallback_count = 0
    qualities = []

    for start in tqdm(
        range(0, len(selected), TEACHER_GENERATION_BATCH_SIZE),
        desc=f"Teacher critique (round {round_idx})",
    ):
        chunk_idx = selected[start:start + TEACHER_GENERATION_BATCH_SIZE]
        chunk_recs = [records[i] for i in chunk_idx]

        raws = [generated.get(rec["record_id"], "") for rec in chunk_recs]
        for rec, raw in zip(chunk_recs, raws):
            rec["student_reasoning"] = raw
            rec["student_answer"] = extract_answer(raw)

        chunk_images = [get_image(rec) for rec in chunk_recs]
        critiques = teacher.refine_reasoning_batch(
            images=chunk_images,
            questions=[rec["question"] for rec in chunk_recs],
            answers=[rec["answer"] for rec in chunk_recs],
            student_reasonings=raws,
        )

        for rec, critique in zip(chunk_recs, critiques):
            q = float(critique.get("quality_score", 0.0) or 0.0)
            qualities.append(q)
            rec["teacher_quality"] = q

            refined_steps = critique.get("refined_steps", [])
            if q >= KEEP_THRESHOLD and refined_steps:
                rec["teacher_refined_reasoning"] = format_steps(refined_steps, rec["answer"])
                rec["target_reasoning"] = rec["teacher_refined_reasoning"]
                rec["target_answer"] = rec["answer"]
                rec["target_source"] = "teacher_refined"
                keep_count += 1
            else:
                if not rec["teacher_seed_reasoning"]:
                    rec["teacher_seed_reasoning"] = minimal_template(rec["answer"])
                rec["target_reasoning"] = rec["teacher_seed_reasoning"]
                rec["target_answer"] = rec["answer"]
                rec["target_source"] = "fallback"
                fallback_count += 1

    stats = {
        "round": round_idx,
        "selected_examples": len(selected),
        "kept_teacher_refinements": keep_count,
        "fallback_to_seed": fallback_count,
        "avg_quality": sum(qualities) / len(qualities) if qualities else 0.0,
        "keep_threshold": KEEP_THRESHOLD,
    }
    return stats


def initialize_warm_start_targets(records: List[Dict[str, Any]], teacher: TeacherVLM) -> Dict[str, Any]:
    n = len(records)
    warm_k = max(WARM_START_MIN_EXAMPLES, int(round(n * WARM_START_FRACTION)))
    warm_k = min(warm_k, n)

    rng = random.Random(RANDOM_SEED)
    warm_indices = sorted(rng.sample(range(n), k=warm_k))
    warm_set = set(warm_indices)

    for rec in tqdm(records, desc="Warm-start defaults"):
        rec["teacher_seed_reasoning"] = minimal_template(rec["answer"])
        rec["target_reasoning"] = rec["teacher_seed_reasoning"]
        rec["target_answer"] = rec["answer"]
        rec["target_source"] = "minimal_template"

    warm_indices_list = list(warm_indices)
    for start in tqdm(
        range(0, len(warm_indices_list), TEACHER_GENERATION_BATCH_SIZE),
        desc="Warm-start teacher generation",
    ):
        chunk_idx = warm_indices_list[start:start + TEACHER_GENERATION_BATCH_SIZE]
        chunk_recs = [records[i] for i in chunk_idx]
        chunk_images = [get_image(rec) for rec in chunk_recs]

        seeds = teacher.generate_seed_reasoning_batch(
            images=chunk_images,
            questions=[rec["question"] for rec in chunk_recs],
            answers=[rec["answer"] for rec in chunk_recs],
        )

        for rec, seed in zip(chunk_recs, seeds):
            rec["teacher_seed_reasoning"] = seed
            rec["target_reasoning"] = seed
            rec["target_answer"] = rec["answer"]
            rec["target_source"] = "warm_start_teacher"

    return {
        "warm_start_examples": warm_k,
        "warm_start_fraction": warm_k / n if n else 0.0,
    }

In [ ]:
# SFT Dataset and Collator

def record_to_messages(rec: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "record_id": rec["record_id"],
        "messages": build_student_messages(
            image=get_image(rec),
            question=rec["question"],
            target_reasoning=rec["target_reasoning"],
            target_answer=rec["target_answer"],
        ),
        "target_source": rec["target_source"],
    }


def record_to_messages_with_aux(rec: Dict[str, Any]) -> Dict[str, Any]:
    """
    Like record_to_messages, but also attaches 'aux_messages': the same
    image/question with the assistant turn built from the model's own
    ORIGINAL (pre-refinement) reasoning -- rec['student_reasoning'] -- used
    for the KL self-distillation term (see USE_KL_DISTILL_LOSS at the top
    of this file, and KLDistillSFTTrainer / _response_kl_divergence below).

    Only records with target_source == 'teacher_refined' AND a non-empty
    student_reasoning actually get a distinct aux target. Everything else
    (fallback targets, warm-start targets, round 0 before any student
    reasoning exists) reuses the primary 'messages' as 'aux_messages' --
    see KLDistillVisionDataCollator's docstring for why that's sufficient
    (the two forward passes become identical, so KL == 0 automatically).
    """
    base = record_to_messages(rec)

    has_real_aux = (
        rec.get("target_source") == "teacher_refined"
        and rec.get("student_reasoning", "").strip() != ""
    )

    if has_real_aux:
        student_answer = rec.get("student_answer", "").strip() or rec["answer"]
        aux_messages = build_student_messages(
            image=get_image(rec),
            question=rec["question"],
            target_reasoning=rec["student_reasoning"],
            target_answer=student_answer,
        )
    else:
        aux_messages = base["messages"]

    base["aux_messages"] = aux_messages
    return base


class VisionDataCollator:
    def __init__(self, model, processor):
        self.base = UnslothVisionDataCollator(model, processor)

    def __call__(self, samples: List[Dict[str, Any]]) -> Dict[str, Any]:
        base_samples = [{"messages": s["messages"]} for s in samples]
        batch = self.base(base_samples)
        return batch


class KLDistillVisionDataCollator:
    """
    Same primary behavior as VisionDataCollator (builds the standard
    input_ids/labels/pixel_values batch from each sample's 'messages',
    i.e. the CE training target), plus a second 'aux_inputs' sub-batch
    built from each sample's 'aux_messages' (the model's own original
    pre-refinement reasoning). KLDistillSFTTrainer.compute_loss pops
    'aux_inputs' off the batch and uses it for the KL self-distillation
    term.

    Requires samples produced by record_to_messages_with_aux (which
    guarantees 'aux_messages' is always present, falling back to the
    primary 'messages' when there's no real aux target this round).
    """

    def __init__(self, model, processor):
        self.base = UnslothVisionDataCollator(model, processor)

    def __call__(self, samples: List[Dict[str, Any]]) -> Dict[str, Any]:
        primary = [{"messages": s["messages"]} for s in samples]
        aux = [{"messages": s["aux_messages"]} for s in samples]

        batch = self.base(primary)
        aux_batch = self.base(aux)

        batch["aux_inputs"] = aux_batch
        return batch


def get_text_hidden_size(model) -> int:
    cfg = getattr(model, "config", None)
    if cfg is None:
        raise ValueError("Model has no config.")

    if hasattr(cfg, "hidden_size") and cfg.hidden_size is not None:
        return int(cfg.hidden_size)

    if hasattr(cfg, "text_config") and hasattr(cfg.text_config, "hidden_size"):
        return int(cfg.text_config.hidden_size)

    if hasattr(cfg, "text_config") and hasattr(cfg.text_config, "dim"):
        return int(cfg.text_config.dim)

    raise ValueError("Could not infer hidden size from model config.")


class SimpleVisionSFTTrainer(SFTTrainer):
    pass

In [ ]:
# KL Self-Distillation

def _response_kl_divergence(
    target_logits: torch.Tensor,
    target_labels: torch.Tensor,
    student_logits: torch.Tensor,
    student_labels: torch.Tensor,
    max_align_tokens: int = KL_MAX_ALIGN_TOKENS,
) -> torch.Tensor:
    
    batch_size = target_logits.size(0)
    device = target_logits.device
    losses = []

    for b in range(batch_size):
        t_mask = target_labels[b] != -100
        s_mask = student_labels[b] != -100
        t_len = int(t_mask.sum().item())
        s_len = int(s_mask.sum().item())
        L = min(t_len, s_len, max_align_tokens)
        if L <= 0:
            continue

        t_idx = t_mask.nonzero(as_tuple=True)[0][:L]
        s_idx = s_mask.nonzero(as_tuple=True)[0][:L]

        t_logits = target_logits[b, t_idx, :]   # [L, V]
        s_logits = student_logits[b, s_idx, :]  # [L, V]

        with torch.no_grad():
            t_log_probs = F.log_softmax(t_logits, dim=-1)
            t_probs = t_log_probs.exp()

        s_log_probs = F.log_softmax(s_logits, dim=-1)
        kl = (t_probs * (t_log_probs - s_log_probs)).sum(dim=-1).mean()
        losses.append(kl)

    if not losses:
        return torch.zeros((), device=device)

    return torch.stack(losses).mean()


def _find_lm_head(model):
    m = model
    seen = set()
    while id(m) not in seen:
        seen.add(id(m))
        if hasattr(m, "lm_head"):
            return m.lm_head
        if hasattr(m, "base_model"):
            m = m.base_model
            continue
        break

    raise RuntimeError(
        "Could not find an `lm_head` attribute while unwrapping PEFT/model "
        f"layers; got stuck at {type(m)}. Print `model` and inspect its "
        "structure, then adjust _find_lm_head to match (e.g. it may need "
        "an extra `.get_base_model()` or a manual `.model` hop for your "
        "PEFT/Unsloth version)."
    )


def raw_logits_forward(model, batch: Dict[str, torch.Tensor]):
    lm_head = _find_lm_head(model)

    forward_kwargs = {k: v for k, v in batch.items() if k != "labels"}
    forward_kwargs["output_hidden_states"] = True
    forward_kwargs["return_dict"] = True

    outputs = model(**forward_kwargs)
    last_hidden = outputs.hidden_states[-1]
    last_hidden = last_hidden.to(lm_head.weight.dtype)
    logits = lm_head(last_hidden)
    return logits


class KLDistillSFTTrainer(SFTTrainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        aux_inputs = inputs.pop("aux_inputs", None)
        outputs = model(**inputs)
        ce_loss = outputs.loss

        if not USE_KL_DISTILL_LOSS or aux_inputs is None:
            return (ce_loss, outputs) if return_outputs else ce_loss

        with torch.no_grad():
            target_logits = raw_logits_forward(model, inputs)

        student_logits = raw_logits_forward(model, aux_inputs)

        kl_loss = _response_kl_divergence(
            target_logits=target_logits,
            target_labels=inputs["labels"],
            student_logits=student_logits,
            student_labels=aux_inputs["labels"],
        )

        total_loss = ce_loss + KL_LOSS_WEIGHT * kl_loss

        try:
            self.log({"ce_loss": float(ce_loss.detach().item()), "kl_loss": float(kl_loss.detach().item())})
        except Exception:
            pass

        return (total_loss, outputs) if return_outputs else total_loss

In [ ]:
# Training

def run_one_sft_round(model, processor, records: List[Dict[str, Any]], round_idx: int) -> None:
    if USE_KL_DISTILL_LOSS:
        converted = [record_to_messages_with_aux(r) for r in records]
        data_collator = KLDistillVisionDataCollator(model, processor)
        trainer_cls = KLDistillSFTTrainer
    else:
        converted = [record_to_messages(r) for r in records]
        data_collator = VisionDataCollator(model, processor)
        trainer_cls = SimpleVisionSFTTrainer

    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16 = torch.cuda.is_available() and not bf16

    if hasattr(FastVisionModel, "for_training"):
        FastVisionModel.for_training(model)
    else:
        model.train()

    set_generation_padding(processor, "right")

    round_dir = os.path.join(OUTPUT_DIR, f"round_{round_idx}")

    resume_from_checkpoint = False
    if os.path.isdir(round_dir):
        existing_ckpts = [d for d in os.listdir(round_dir) if d.startswith("checkpoint-")]
        if existing_ckpts:
            resume_from_checkpoint = True
            print(f"Found existing step checkpoint(s) in {round_dir} -- resuming mid-round training.")

    trainer = trainer_cls(
        model=model,
        tokenizer=processor,
        data_collator=data_collator,
        train_dataset=converted,
        args=SFTConfig(
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            max_steps=MAX_STEPS_PER_ROUND,
            learning_rate=LR,
            warmup_steps=WARMUP_STEPS,
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=WEIGHT_DECAY,
            lr_scheduler_type="cosine",
            output_dir=round_dir,
            report_to="none",
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_seq_length=MAX_SEQ_LENGTH,
            bf16=bf16,
            fp16=fp16,
            dataloader_num_workers=2,
            dataloader_pin_memory=True,
            save_strategy="steps",
            save_steps=SAVE_STEPS_PER_ROUND,
            save_total_limit=SAVE_TOTAL_LIMIT_PER_ROUND,
        ),
    )
    print(f"\n--- Training round {round_idx} (KL distill loss {'ON' if USE_KL_DISTILL_LOSS else 'OFF'}) ---")
    trainer.train(resume_from_checkpoint=resume_from_checkpoint)

In [ ]:
# Evaluation

@torch.no_grad()
def evaluate_split(model, processor, records: List[Dict[str, Any]], split_name: str) -> Dict[str, Any]:
    if hasattr(FastVisionModel, "for_inference"):
        FastVisionModel.for_inference(model)
    else:
        model.eval()

    set_generation_padding(processor, "left")

    total = 0
    correct = 0

    closed_total = 0
    closed_correct = 0

    open_total = 0
    open_recall_sum = 0.0
    open_bleu1_sum = 0.0
    open_rouge1_sum = 0.0
    open_rouge2_sum = 0.0
    open_rougeL_sum = 0.0

    step_counts: List[int] = []

    for rec in tqdm(records, desc=f"Evaluating {split_name}"):
        img = get_image(rec)
        msgs = build_student_messages(img, rec["question"])
        prompt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(
            text=[prompt],
            images=[img],
            return_tensors="pt",
            padding=True,
        )
        for k, v in list(inputs.items()):
            if torch.is_tensor(v):
                inputs[k] = v.to(model.device)

        gen_kwargs = dict(max_new_tokens=GENERATION_MAX_NEW_TOKENS, do_sample=False)
        out_ids = model.generate(**inputs, **gen_kwargs)
        input_len = inputs["input_ids"].shape[1]
        decoded = processor.decode(out_ids[0][input_len:], skip_special_tokens=True).strip()

        pred_answer = extract_answer(decoded)
        gt_answer = rec["answer"]
        step_counts.append(len(extract_steps(decoded)))

        total += 1

        answer_type = rec.get("answer_type") or ("CLOSED" if is_closed_answer(gt_answer) else "OPEN")
        mode = "closed" if answer_type == "CLOSED" else "open"

        exact = exact_match(pred_answer, gt_answer)
        correct += exact

        if mode == "closed":
            closed_total += 1
            closed_correct += exact
        else:
            open_total += 1
            recall = token_overlap_recall(pred_answer, gt_answer)
            bleu1 = compute_bleu1(pred_answer, gt_answer)
            r1, r2, rL = compute_rouge(pred_answer, gt_answer)
            open_recall_sum += recall
            open_bleu1_sum += bleu1
            open_rouge1_sum += r1
            open_rouge2_sum += r2
            open_rougeL_sum += rL

    set_generation_padding(processor, "right")

    if hasattr(FastVisionModel, "for_training"):
        FastVisionModel.for_training(model)
    else:
        model.train()

    result = {
        "split": split_name,
        "total": total,
        "accuracy": correct / total if total else 0.0,
        "mean_reasoning_steps": sum(step_counts) / len(step_counts) if step_counts else 0.0,
        "closed_total": closed_total,
        "closed_accuracy": (closed_correct / closed_total) if closed_total else None,
        "open_total": open_total,
        "open_recall": (open_recall_sum / open_total) if open_total else None,
        "open_bleu1": (open_bleu1_sum / open_total) if open_total else None,
        "open_rouge1": (open_rouge1_sum / open_total) if open_total else None,
        "open_rouge2": (open_rouge2_sum / open_total) if open_total else None,
        "open_rougeL": (open_rougeL_sum / open_total) if open_total else None,
    }
    return result

In [ ]:
# Main

set_seed(RANDOM_SEED)
make_output_dirs()

training_start = time.time()

state = load_state()
print(f"Resume state loaded: {state}")

print("Downloading/locating SLAKE files (train.json, test.json, imgs.zip)...")
slake_paths = download_slake_files(SLAKE_LOCAL_DIR)

# --- train records: reuse cached (already-targeted) records if warm start
# already completed in a prior run; otherwise (re)build from the SLAKE JSON.
if state["warm_start_done"]:
    cached = load_train_records_state()
    if cached is not None:
        train_records = cached
        print(f"Resumed {len(train_records)} cached train records (warm start already done).")
    else:
        print("state.json says warm_start_done=True but the records cache is missing -- rebuilding from scratch.")
        state["warm_start_done"] = False
        train_records = build_records(slake_paths[SLAKE_TRAIN_JSON], slake_paths["imgs_dir"], max_rows=MAX_TRAIN_SAMPLES)
else:
    train_records = build_records(slake_paths[SLAKE_TRAIN_JSON], slake_paths["imgs_dir"], max_rows=MAX_TRAIN_SAMPLES)

test_records = build_records(slake_paths[SLAKE_TEST_JSON], slake_paths["imgs_dir"], max_rows=MAX_TEST_SAMPLES)
print(f"Train records (English QA pairs): {len(train_records)}")
print(f"Test records (English QA pairs) : {len(test_records)}")

print("Loading teacher model...")
teacher = TeacherVLM(TEACHER_MODEL_ID)

if not state["warm_start_done"]:
    print("Initializing warm-start targets...")
    warm_stats = initialize_warm_start_targets(train_records, teacher)
    log_jsonl(LOG_PATH, {"event": "warm_start", **warm_stats})
    print(f"Warm-start examples: {warm_stats['warm_start_examples']} ({warm_stats['warm_start_fraction']:.2%})")

    state["warm_start_done"] = True
    save_train_records(train_records)
    save_state(state)
else:
    print("Warm-start already completed in a prior run -- skipping.")

# --- student model: resume from the highest completed round's checkpoint
# if one exists, otherwise start from the base model with a fresh LoRA.
completed_rounds = sorted(state.get("completed_rounds", []))
last_completed_round = completed_rounds[-1] if completed_rounds else -1

print("Loading student model...")
if last_completed_round >= 0:
    resume_ckpt = os.path.join(ROUND_CHECKPOINT_DIR, f"round_{last_completed_round}")
    print(f"Resuming student model + LoRA adapter from: {resume_ckpt}")
    model, processor = FastVisionModel.from_pretrained(
        resume_ckpt,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )
    disable_max_length_conflict(model)
else:
    model, processor = FastVisionModel.from_pretrained(
        STUDENT_MODEL_ID,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )
    disable_max_length_conflict(model)

    model = FastVisionModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_R * 2,
        lora_dropout=0.0,
        bias="none",
        random_state=RANDOM_SEED,
        use_gradient_checkpointing="unsloth",
        finetune_vision_layers=True,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
    )

os.makedirs(ROUND_CHECKPOINT_DIR, exist_ok=True)

for round_idx in range(NUM_SELF_TRAIN_ROUNDS):
    print("\n" + "=" * 80)
    print(f"SELF-TRAINING ROUND {round_idx}")
    print("=" * 80)

    if round_idx in completed_rounds:
        print(f"Round {round_idx} already fully completed in a prior run -- skipping entirely.")
        continue

    if round_idx > 0:
        if state.get("refine_done_round", -1) >= round_idx:
            print(f"Round {round_idx} teacher-refined targets already cached -- reusing them.")
        else:
            stats = refine_training_targets(
                records=train_records,
                model=model,
                processor=processor,
                teacher=teacher,
                round_idx=round_idx,
            )
            log_jsonl(LOG_PATH, {"event": "refine", **stats})
            print(
                f"Refine stats: kept={stats['kept_teacher_refinements']} "
                f"fallback={stats['fallback_to_seed']} "
                f"avg_quality={stats['avg_quality']:.4f}"
            )
            state["refine_done_round"] = round_idx
            save_train_records(train_records)
            save_state(state)
    else:
        print("Round 0: warm-start SFT on teacher seed chains + minimal templates.")
        print("(No KL term this round -- student_reasoning doesn't exist yet.)")

    run_one_sft_round(model, processor, train_records, round_idx)

    round_ckpt = os.path.join(ROUND_CHECKPOINT_DIR, f"round_{round_idx}")
    model.save_pretrained(round_ckpt)
    processor.save_pretrained(round_ckpt)
    print(f"Saved checkpoint to {round_ckpt}")

    completed_rounds.append(round_idx)
    state["completed_rounds"] = sorted(completed_rounds)
    save_state(state)

final_path = os.path.join(OUTPUT_DIR, "final_model")
if not state.get("final_eval_done", False) or not os.path.isdir(final_path):
    model.save_pretrained(final_path)
    processor.save_pretrained(final_path)
    print(f"Saved final model to {final_path}")

print("\n" + "=" * 80)
print("FINAL EVALUATION (after all self-training rounds)")
print("=" * 80)

if state.get("final_eval_done", False):
    print("Final evaluation already completed in a prior run -- re-running anyway for a fresh log entry.")

eval_result = evaluate_split(model, processor, test_records, "test")
log_jsonl(EVAL_PATH, {"round": "final", **eval_result})
print(
    f"Final eval: "
    f"acc={eval_result['accuracy']:.4f}, "
    f"mean_steps={eval_result['mean_reasoning_steps']:.2f}"
)
if eval_result["closed_accuracy"] is not None:
    print(f"  closed_accuracy={eval_result['closed_accuracy']:.4f} (n={eval_result['closed_total']})")
if eval_result["open_recall"] is not None:
    print(
        f"  open_recall={eval_result['open_recall']:.4f} "
        f"open_bleu1={eval_result['open_bleu1']:.4f} "
        f"open_rouge1={eval_result['open_rouge1']:.4f} "
        f"open_rouge2={eval_result['open_rouge2']:.4f} "
        f"open_rougeL={eval_result['open_rougeL']:.4f} "
        f"(n={eval_result['open_total']})"
    )

state["final_eval_done"] = True
save_state(state)

total_seconds = time.time() - training_start
hrs = int(total_seconds // 3600)
mins = int((total_seconds % 3600) // 60)
secs = total_seconds % 60

summary = (
    f"THIS RUN'S WALLCLOCK TIME: {hrs}h {mins}m {secs:.1f}s "
    f"({total_seconds:.1f} seconds) -- note this only covers the current "
    f"process; if the script was resumed from a checkpoint, add the time "
    f"spent in earlier interrupted runs to get the true total."
)
print("\n" + "=" * 80)
print(summary)
print("=" * 80)

with open(os.path.join(OUTPUT_DIR, "total_finetune_time.txt"), "a", encoding="utf-8") as f:
    f.write(summary + "\n")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Resume state loaded: {'warm_start_done': False, 'refine_done_round': -1, 'completed_rounds': [], 'final_eval_done': False}
Downloading/locating SLAKE files (train.json, test.json, imgs.zip)...


train.json:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/636k [00:00<?, ?B/s]

imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

...

Refine stats: kept=4620 fallback=298 avg_quality=0.9399
Unsloth: Model does not have a default image size - using 512

--- Training round 1 (KL distill loss ON) ---
Unsloth: Will smartly offload gradients to save VRAM!
{'ce_loss': '0.3333', 'kl_loss': '0.000103', 'epoch': 0}
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
{'ce_loss': '0.2448', 'kl_loss': '6.676e-05', 'epoch': 0}
{'ce_loss': '0.2212', 'kl_loss': '0.0001984', 'epoch': 0}
{'ce_loss': '0.2658', 'kl_loss': '0.0001278', 'epoch': 0}
{'loss': '1.065', 'grad_norm': '6.168', 'learning_rate': '0', 'epoch': '0.003252'}
{'ce_loss': '0.3856', 'kl_loss': '0.0001163', 'epoch': '0.003252'}
{'ce_loss': '0.331', 'kl_loss': '0.0001926', 'epoch': '0.003252'}
{'ce_loss': '0.412', 'kl_loss': '0.0001087', 'epoch': '0.003252'}
{'ce_loss': '0.4767', 'kl_loss': '0.0001841', 'epoch': '0.003252'}
{'loss': '1.606', 'grad_norm': '10.8', 'learning_rate': '4e-05', 'epoch': '0.006504'}
{'ce_loss': '0.5146', 'kl_loss': '0.0

Teacher critique (round 2): 100%|██████████| 615/615 [32:46<00:00,  3.20s/it]


Refine stats: kept=4678 fallback=240 avg_quality=0.9513
Unsloth: Model does not have a default image size - using 512

--- Training round 2 (KL distill loss ON) ---
Unsloth: Will smartly offload gradients to save VRAM!
{'ce_loss': '0.1435', 'kl_loss': '0.0001526', 'epoch': 0}
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
{'ce_loss': '0.06146', 'kl_loss': '2.062e-05', 'epoch': 0}
{'ce_loss': '0.08506', 'kl_loss': '0.0001621', 'epoch': 0}
{'ce_loss': '0.04836', 'kl_loss': '7.868e-05', 'epoch': 0}
{'loss': '0.3386', 'grad_norm': '1.018', 'learning_rate': '0', 'epoch': '0.003252'}
{'ce_loss': '0.07082', 'kl_loss': '7.439e-05', 'epoch': '0.003252'}
{'ce_loss': '0.06366', 'kl_loss': '4.053e-05', 'epoch': '0.003252'}
{'ce_loss': '0.06114', 'kl_loss': '0.0001249', 'epoch': '0.003252'}
{'ce_loss': '0.05644', 'kl_loss': '5.603e-05', 'epoch': '0.003252'}
{'loss': '0.2522', 'grad_norm': '1.65', 'learning_rate': '4e-05', 'epoch': '0.006504'}
{'ce_loss': '0.1125', 'kl

Evaluating test: 100%|██████████| 1061/1061 [1:40:49<00:00,  5.70s/it]

Final eval: acc=0.6993, mean_steps=1.69
  closed_accuracy=0.7933 (n=416)
  open_recall=0.7139 open_bleu1=0.6987 open_rouge1=0.7089 open_rouge2=0.1608 open_rougeL=0.7056 (n=645)

THIS RUN'S WALLCLOCK TIME: 5h 12m 32.1s (18752.1 seconds) -- note this only covers the current process; if the script was resumed from a checkpoint, add the time spent in earlier interrupted runs to get the true total.
